In [34]:
from utils.generic_utils import load_all_games_csv, basic_win_prob_for_et, predict_lr, get_teams
from utils.elo_tracker_utils import evaluate_elo_prob_func
from elos.elo_tracker import EloTracker
from scipy.special import expit
import numpy as np

# Win Probability Analysis

This notebook will compare several different methods to estimate win probabilities from Elo ratings, and possibly home advantage, travel distance, and rest days.

## Get all Games

In [ ]:
all_games = load_all_games_csv('../data/gameinfo_cleaned.csv', preprocess=True)
#all_games = all_games[(all_games['season'] >= 1990) & (all_games['season'] < 2000)]
all_games.head()

/Users/lancehendricks/Documents/College Coding/ML/Elo Ratings/analysis/src/utils/generic_utils.py:30: DtypeWarning: Columns (10,11,13,17,19,20,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  all_games = pd.read_csv(filename)


,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,...,homerestdays,visrestdays,homepitcherrgs,vispitcherrgs,hometeamrgs,visteamrgs,homepitcherminusteamrgs,vispitcherminusteamrgs,homelastkwinpct,vislastkwinpct
gid,,,,,,,,,,,,,,,,,,,,,
CIN189804150,CL4,CIN,CIN05,18980415,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
LS3189804150,PIT,LS3,LOU03,18980415,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
SLN189804150,CHN,SLN,STL05,18980415,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
BLN189804160,WSN,BLN,BAL07,18980416,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
CIN189804160,CL4,CIN,CIN05,18980416,0.0,0:00PM,day,NaN,NaN,False,...,1.0,1.0,39.0,39.0,65.9,64.9,-26.9,-25.9,1.0,0.0


## Evaluate Simple Probability model

In [39]:
bce, accuracy = evaluate_elo_prob_func(all_games, basic_win_prob_for_et, K=3, use_margin_of_victory=True, skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6794833765075686
Accuracy: 0.5651411521689236


## With +28 Adjustment for Home Team

In [42]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game_info: basic_win_prob_for_et(home_elo + 28, away_elo, game_info), K=2, use_margin_of_victory=True, skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6764792626452175
Accuracy: 0.5720817075969704


## With + 1.9% Adjustment for Home Team

In [43]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game_info: basic_win_prob_for_et(home_elo*1.019, away_elo, game_info), K=2, use_margin_of_victory=True, skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6764717794260093
Accuracy: 0.5718108790452145


## With Logistic Regression

In [53]:
# First need to fit using some Elos - use the initial basic probability func.

df_to_fit = all_games.copy()
et = EloTracker(K=3, elo_prob_func=basic_win_prob_for_et, use_margin_of_victory=True)
et.add_history(df_to_fit, add_elos_to_df=True)

df_to_fit = df_to_fit.dropna(subset=['homedistancetraveled', 'visdistancetraveled', 'homerestdays', 'visrestdays', 'homepitcherminusteamrgs', 'vispitcherminusteamrgs'])


In [54]:
df_to_fit['elodiff'] = df_to_fit['viselobefore'] - df_to_fit['homeelobefore']
df_to_fit['restdiff'] = df_to_fit['visrestdays'] - df_to_fit['homerestdays']
df_to_fit['distancediff'] = df_to_fit['visdistancetraveled'] - df_to_fit['homedistancetraveled']
df_to_fit['homediff'] =  0 - 1
df_to_fit['pitcherdiff'] = df_to_fit['vispitcherminusteamrgs'] - df_to_fit['homepitcherminusteamrgs']
# df_to_fit['momentumdiff'] = df_to_fit['vismomentum'] - df_to_fit['homemomentum']

#features = ['elodiff', 'distancediff', 'restdiff']
features = ['elodiff', 'homediff', 'restdiff', 'distancediff', 'pitcherdiff']
X = df_to_fit[features].to_numpy()
y = df_to_fit['homewon'].astype(int).to_numpy().reshape(-1,1)

s = -np.log(10) / 400

In [55]:
# Fit via GD
w = np.zeros((5,1))
w[0,0] = s # Becomes 1 once dividing by s

step = 0.01 # Slightly higher for small gradients
iterations = 10000

for _ in range(iterations):

    z = X @ w
    y_hat = expit(z)
    
    w_grad = (1/X.shape[0]) * X.T @ (y_hat - y)
    
    w_grad[0,0] = 0
    
    #print(w_grad)
    
    w = w - step*w_grad
    
w

array([[-0.00575646],
       [-0.15879801],
       [-0.02854395],
       [ 0.00147698],
       [-0.00909051]])

In [56]:
# Convert back to interpretable coefficients for individual Elo adjustments
w = (1/s) * w
w

array([[ 1.        ],
       [27.58603899],
       [ 4.95859171],
       [-0.25657825],
       [ 1.57918407]])

In [57]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w), K=3, use_margin_of_victory=True, skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6753916623142618
Accuracy: 0.574514574248336
